# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue is the priority score from Logistic Regression (w05/w06), ranked highest to lowest, with a short **reason code** attached to each page — plain, checkable statements built from the same features the model leans on (w05's permutation importance: visibility consistency, real-user traffic, position, staleness), not a black-box score alone. A human reviewer should be able to look at a page's reason code and immediately see why it surfaced, without opening the model.

In [8]:
import os

REPO_URL = "https://github.com/AnaraHayat/flyrank_assignment1.git"
REPO_DIR = "/content/flyrank_assignment1"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

import numpy as np
import pandas as pd
from pathlib import Path

DATA_REL = "data/raw/content_refresh_anonymized.csv"
start = Path.cwd()
repo_root = None
for candidate in [start, *start.parents]:
    if (candidate / DATA_REL).exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError(f"Couldn't find {DATA_REL} above {start}. Run the git-clone cell first if this is Colab.")
os.chdir(repo_root)

df = pd.read_csv(DATA_REL)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].values

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = ["competition_level", "content_type", "main_intent"]
missing_prone = ["search_volume", "competition", "cpc", "word_count", "char_count"]
for c in missing_prone:
    df[f"has_{c}"] = df[c].notna().astype(int)
missing_flag_features = [f"has_{c}" for c in missing_prone]

X = df[numeric_features + categorical_features + missing_flag_features].copy()
for c in numeric_features + missing_flag_features:
    X[c] = X[c].fillna(0)
for c in categorical_features:
    X[c] = X[c].fillna("unknown")

groups = df["client_id"]
prep = ColumnTransformer([
    ("num", StandardScaler(), numeric_features + missing_flag_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

# PRODUCTION model: fit on every labeled row we have, so the deployed queue uses all available evidence.
# (Distinct from the w05/w06 HOLDOUT model, which is deliberately trained on 75% of clients only, so its
#  P@50 = 0.78 measures honest generalization -- that number is what we quote as the model's real skill.
#  This queue is what a team would actually run each week.)
lr_production = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42))])
lr_production.fit(X, y)
df["priority_score"] = lr_production.predict_proba(X)[:, 1]

print(f"Scored {len(df)} pages across {df['client_id'].nunique()} clients.")

Scored 30000 pages across 32 clients.


In [9]:
# Reason codes: plain, checkable statements built from the features the model actually leans on
# (per w05 permutation importance: visibility consistency, real-user traffic, position, staleness).
med_days_with_impr = df["days_with_impressions"].median()
med_users = df["users_90d"].median()
med_pos = df.loc[df["avg_position"] > 0, "avg_position"].median()

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180:
        reasons.append("stale: 180+ days since last update")
    if row["days_with_impressions"] < med_days_with_impr:
        reasons.append("inconsistent visibility: below-median days with impressions in the last 90d")
    if row["avg_position"] > med_pos and row["avg_position"] > 0:
        reasons.append("weak position: average position past the site's typical page")
    if row["users_90d"] < med_users:
        reasons.append("low real-user traffic: below-median users_90d")
    if not reasons:
        reasons.append("elevated by several moderate signals together, no single dominant flag")
    return "; ".join(reasons[:3])

df["reason_codes"] = df.apply(reason_codes, axis=1)

ranked_queue = df.sort_values("priority_score", ascending=False).reset_index(drop=True)
ranked_queue.insert(0, "queue_rank", np.arange(1, len(ranked_queue) + 1))

print("Top 10 of the ranked refresh-priority queue:")
print(ranked_queue[["queue_rank", "content_id", "client_id", "priority_score", "reason_codes"]].head(10).to_string(index=False))

Top 10 of the ranked refresh-priority queue:
 queue_rank           content_id         client_id  priority_score                                                                reason_codes
          1 content_4560b0a818ab client_19581e27de        1.000000 inconsistent visibility: below-median days with impressions in the last 90d
          2 content_8e7ba84a972b client_7f2253d7e2        0.999922      elevated by several moderate signals together, no single dominant flag
          3 content_c8ad1f4d0e56 client_7f2253d7e2        0.998199      elevated by several moderate signals together, no single dominant flag
          4 content_a22b7f6c73c5 client_7f2253d7e2        0.996376      elevated by several moderate signals together, no single dominant flag
          5 content_70b8f5323e29 client_7f2253d7e2        0.984409                weak position: average position past the site's typical page
          6 content_c53566c0e1b1 client_7f2253d7e2        0.981621      elevated by several moder

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** A content ops or SEO team pulls the top N rows of this queue (N sized to weekly review capacity — 20-50 pages is realistic) each week and treats it as a **starting shortlist for human review**, not an auto-refresh trigger. The reason codes tell a reviewer what to check first.

**Who this is for.** Teams with a similar shape of portfolio to the 32 clients in this snapshot — content with 90 days of Google Search Console history, a mix of client sizes. It is not validated for single-page sites, brand-new accounts with no 90-day history, or non-search-driven content (e.g., paid-only landing pages).

**Where it stops being valid:**
- **A brand-new client with no holdout evidence.** Section 2 of `w06_validation_audit.ipynb` measured P@50 = 0.78 on 8 *held-out* clients — that's the honest generalization number. It is not 0.94 (the number an ungrouped split would have reported); a new client should be expected closer to the lower, honest figure.
- **This exact snapshot only.** The label (`is_declining_label`) is a 30-day-vs-previous-30-day comparison at one point in time. The queue should be **re-scored on a fresh pull**, not reused from an old export.
- **Correlation, not cause.** Nothing here says *why* a page is declining — only that its profile resembles other pages that did. A reviewer still has to look at the actual page.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on a queue row, a human should check:**
1. Open the actual page — does the content still match current search intent, or has the topic moved on?
2. Confirm the decline isn't explained by something outside content quality (e.g. a known algorithm update, a site migration, a broken canonical/redirect) — none of that is in this feature set.
3. Check whether the page overlaps with another page on the site (cannibalization) — refreshing the wrong one of two competing pages wastes the effort.
4. Sanity-check the reason code against the raw numbers for that row — the code is a summary, not a proof.

**No-go list — never automate:**
- **Auto-publishing content changes.** The model flags *candidates for review*; it says nothing about what a rewrite should say.
- **Auto-deprioritizing or removing pages** on the model's say-so alone — a false positive here just means a person double-checks a healthy page; a false-positive *removal* is a real, harder-to-reverse loss.
- **Using this score as a performance metric for content writers.** It is a triage tool for a portfolio, not a judgment of any individual piece or its author.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [10]:
# A lightweight, checkable version of what a monitoring dashboard would track over time.
# These are the numbers a team would watch snapshot-over-snapshot; this cell just documents
# the current baseline reading so future runs have something to compare against.

baseline_reading = {
    "snapshot_base_rate": round(float(y.mean()), 3),           # share of pages currently labeled declining
    "queue_top50_share_declining": round(
        float((ranked_queue.head(50)["is_declining_label"] == 1).mean()), 3
    ),                                                            # precision of today's top-50 (in-sample, production model)
    "n_clients": int(df["client_id"].nunique()),
    "n_pages": int(len(df)),
}
print(baseline_reading)

{'snapshot_base_rate': 0.542, 'queue_top50_share_declining': 0.88, 'n_clients': 32, 'n_pages': 30000}


**Retrain / re-check triggers:**
- **Base rate drift.** If the share of pages labeled `declining` moves far from today's reading (~0.54), the portfolio's behavior has shifted enough that the old model's calibration may no longer hold.
- **Queue precision drop.** If a team starts tracking real outcomes on queue picks (did the flagged page actually keep declining after 30 more days?) and that precision drifts well below the P@50 = 0.78 holdout figure, that's a concrete retrain signal — not a guess.
- **New clients added.** Since P@50 was only validated on 8 held-out clients, every batch of genuinely new clients is itself a mini validation test — track their queue precision separately for the first cycle.
- **Time elapsed.** This snapshot covers one 90-day window; SEO and search-behavior patterns shift with algorithm updates and seasonality, so a stale model (no retrain in 2+ quarters) should be treated as suspect even without a specific triggering event.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
from pathlib import Path

outputs_dir = Path("work/outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

# 1) The full ranked queue -- public-safe (hashed content_id / client_id only, no names or URLs)
export_cols = [
    "queue_rank", "content_id", "client_id", "priority_score", "reason_codes",
    "is_declining_label", "content_type", "main_intent",
]
ranked_queue[export_cols].to_csv(outputs_dir / "w07_ranked_queue.csv", index=False)

# 2) Top-50 slice -- the realistic weekly worklist, small enough to show in the paper as a table
ranked_queue[export_cols].head(50).to_csv(outputs_dir / "w07_ranked_queue_top50.csv", index=False)

# 3) Monitoring baseline reading -- so a future run can diff against today
import json
with open(outputs_dir / "w07_monitoring_baseline.json", "w") as f:
    json.dump(baseline_reading, f, indent=2)

print("Wrote:")
for p in sorted(outputs_dir.glob("w07_*")):
    print(" -", p)

Wrote:
 - work/outputs/w07_monitoring_baseline.json
 - work/outputs/w07_ranked_queue.csv
 - work/outputs/w07_ranked_queue_top50.csv


## Self-check

Before you submit, confirm each line honestly:

- [ Done ] Every section above is filled — markdown thinking AND the code that backs it
- [Done] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done ] No client names, URLs, or private queries anywhere
- [Done ] My claims use careful words: observed, measured, directional, decision-support
- [ Done] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.